# Lakehouse de vendas com Spark Structured Streaming

**Arquitetura Medallion sem Delta Lake** | Bronze: Python/PySpark | Silver e Gold: Spark SQL | Armazenamento: Parquet

**Escala:** 1.000.000 vendas, 250.000 clientes e 100.000 produtos. **Execução:** Jupyter local ou Docker com PySpark 3.5+ / 4.x e Java compatível.

> Este é um laboratório didático: o produtor cria arquivos JSON em micro-lotes; o Structured Streaming os consome com checkpoint. O processamento das transformações Silver e Gold usa SQL em `spark.sql()`, orquestrado pelo Python. Sem Delta/Iceberg/Hudi, Parquet não oferece transações ACID, `MERGE` ou atualizações atômicas.


## 1. Preparação
Instale `pyspark` no ambiente do kernel (`%pip install pyspark`) caso necessário. Reinicie o kernel após a instalação. Ajuste `BASE` para um diretório com espaço livre (recomendado: alguns GB). O laboratório roda em modo local e pode exigir vários minutos.


In [1]:
from pathlib import Path
import json, time, shutil, uuid
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder.appName("MedallionVendasStreamingSemDelta")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
BASE = Path("./lakehouse_vendas_streaming").resolve()
LANDING = BASE / "landing" / "vendas"
BRONZE, SILVER, GOLD = (BASE / x for x in ("bronze", "silver", "gold"))
CHECKPOINT = BASE / "checkpoints" / "vendas"
for p in [LANDING, BRONZE, SILVER, GOLD, CHECKPOINT]: p.mkdir(parents=True, exist_ok=True)
print("Spark:", spark.version, "| Lakehouse:", BASE)


Spark: 3.5.0 | Lakehouse: /home/jovyan/work/lakehouse_vendas_streaming


## 2. Contratos de dados e dimensões
Os identificadores são determinísticos. Dimensões de clientes e produtos são geradas uma única vez em Parquet. O produtor gera vendas com `spark.range`, sem criar um milhão de objetos Python em memória.


In [2]:
N_VENDAS = 1_000_000
N_CLIENTES = 250_000
N_PRODUTOS = 100_000
TAMANHO_LOTE = 50_000

clientes_path = str(BRONZE / "clientes")
produtos_path = str(BRONZE / "produtos")
if not (BRONZE / "clientes").exists() or not list((BRONZE / "clientes").glob("*.parquet")):
    (spark.range(1, N_CLIENTES + 1).select(
        F.col("id").cast("long").alias("cliente_id"),
        F.concat(F.lit("Cliente "), F.col("id")).alias("nome_cliente"),
        F.expr("element_at(array('SP','RJ','MG','PR','RS'), cast(id % 5 + 1 as int))").alias("uf")
    ).write.mode("overwrite").parquet(clientes_path))
if not (BRONZE / "produtos").exists() or not list((BRONZE / "produtos").glob("*.parquet")):
    (spark.range(1, N_PRODUTOS + 1).select(
        F.col("id").cast("long").alias("produto_id"),
        F.concat(F.lit("Produto "), F.col("id")).alias("nome_produto"),
        F.expr("element_at(array('Eletronicos','Casa','Moda','Esporte'), cast(id % 4 + 1 as int))").alias("categoria")
    ).write.mode("overwrite").parquet(produtos_path))
print("Dimensões gravadas")


Dimensões gravadas


## 3. Produtor de arquivos de vendas (Python)
Cada arquivo de lote é publicado numa pasta temporária e movido para a pasta monitorada após a escrita. Para **simular chegada a cada 10 segundos**, configure `INTERVALO_SEGUNDOS = 10`; para demonstração rápida, deixe `0`. Execute o produtor **uma única vez** por ambiente ou use a célula de limpeza para reiniciar.


In [3]:
INTERVALO_SEGUNDOS = 0  # altere para 10 para simular ingestão periódica

def gerar_lotes(inicio=0, fim=N_VENDAS, tamanho=TAMANHO_LOTE, intervalo=INTERVALO_SEGUNDOS):
    for a in range(inicio, fim, tamanho):
        b = min(a + tamanho, fim)
        destino = LANDING / f"lote_{a:07d}_{b:07d}.json"
        if destino.exists():
            print("Já existe:", destino.name)
            continue
        df = (spark.range(a, b)
              .select((F.col("id") + 1).cast("long").alias("venda_id"),
                      (F.col("id") % N_CLIENTES + 1).cast("long").alias("cliente_id"),
                      (F.col("id") * 7 % N_PRODUTOS + 1).cast("long").alias("produto_id"),
                      F.expr("cast(id % 5 + 1 as int)").alias("quantidade"),
                      F.round(F.lit(10.0) + (F.col("id") % 999) * F.lit(0.37), 2).alias("preco_unitario"),
                      F.date_format(F.expr("timestampadd(MINUTE, cast(id % 525600 as int), timestamp'2025-01-01 00:00:00')"), "yyyy-MM-dd'T'HH:mm:ss").alias("data_venda")))
        # Spark escreve um diretório; compactamos cada micro-lote em um arquivo JSONL.
        temp = BASE / "staging" / f"{a}_{uuid.uuid4().hex}"
        df.coalesce(1).write.mode("overwrite").json(str(temp))
        parte = next(temp.glob("part-*.json"))
        parte.replace(destino)  # movimentação atômica em um mesmo filesystem local
        shutil.rmtree(temp)
        print(f"Publicado: {destino.name} ({b-a:,} vendas)")
        if intervalo and b < fim: time.sleep(intervalo)

# Primeiro publique os lotes; a consulta de streaming os processará a seguir.
gerar_lotes()


Publicado: lote_0000000_0050000.json (50,000 vendas)
Publicado: lote_0050000_0100000.json (50,000 vendas)
Publicado: lote_0100000_0150000.json (50,000 vendas)
Publicado: lote_0150000_0200000.json (50,000 vendas)
Publicado: lote_0200000_0250000.json (50,000 vendas)
Publicado: lote_0250000_0300000.json (50,000 vendas)
Publicado: lote_0300000_0350000.json (50,000 vendas)
Publicado: lote_0350000_0400000.json (50,000 vendas)
Publicado: lote_0400000_0450000.json (50,000 vendas)
Publicado: lote_0450000_0500000.json (50,000 vendas)
Publicado: lote_0500000_0550000.json (50,000 vendas)
Publicado: lote_0550000_0600000.json (50,000 vendas)
Publicado: lote_0600000_0650000.json (50,000 vendas)
Publicado: lote_0650000_0700000.json (50,000 vendas)
Publicado: lote_0700000_0750000.json (50,000 vendas)
Publicado: lote_0750000_0800000.json (50,000 vendas)
Publicado: lote_0800000_0850000.json (50,000 vendas)
Publicado: lote_0850000_0900000.json (50,000 vendas)
Publicado: lote_0900000_0950000.json (50,000 v

## 4. Bronze — ingestão incremental em Python
O Structured Streaming observa a pasta JSON, adiciona metadados de ingestão e persiste em Parquet. `maxFilesPerTrigger=2` limita o tamanho de cada micro-lote. `availableNow=True` processa o backlog e encerra; para fluxo contínuo, substitua por `processingTime="10 seconds"` e mantenha o produtor rodando em outro processo.


In [4]:
schema_vendas = T.StructType([
    T.StructField("venda_id", T.LongType()),
    T.StructField("cliente_id", T.LongType()),
    T.StructField("produto_id", T.LongType()),
    T.StructField("quantidade", T.IntegerType()),
    T.StructField("preco_unitario", T.DoubleType()),
    T.StructField("data_venda", T.StringType()),
])

fonte = (spark.readStream.schema(schema_vendas)
         .option("maxFilesPerTrigger", 2)
         .json(str(LANDING)))
bronze_stream = (fonte.withColumn("ingestao_em", F.current_timestamp())
                 .withColumn("arquivo_origem", F.input_file_name()))

consulta_bronze = (bronze_stream.writeStream
    .format("parquet").outputMode("append")
    .option("path", str(BRONZE / "vendas"))
    .option("checkpointLocation", str(CHECKPOINT / "bronze"))
    .trigger(availableNow=True).start())
consulta_bronze.awaitTermination()
print("Bronze:", spark.read.parquet(str(BRONZE / "vendas")).count())


Bronze: 1000000


## 5. Silver — limpeza, validação e enriquecimento com Spark SQL
A camada Silver aplica tipagem, deduplicação por `venda_id`, validações e relacionamento com as dimensões. Registros inválidos vão para quarentena. O processamento SQL lê a fotografia completa da Bronze para simplificar a correção de reprocessamentos em um laboratório; em produção, seria recomendável usar tabelas transacionais e CDC.

**Importante:** a Silver é recriada em uma pasta de estágio e publicada após a escrita, evitando leitura e sobrescrita simultâneas no mesmo caminho. A troca de diretórios não representa uma transação ACID.


In [5]:
def publicar_parquet(df, destino: Path):
    tmp = BASE / "staging" / f"publish_{uuid.uuid4().hex}"
    tmp.parent.mkdir(parents=True, exist_ok=True)
    df.write.mode("overwrite").parquet(str(tmp))
    anterior = destino.parent / f"_backup_{destino.name}_{uuid.uuid4().hex}"
    if destino.exists(): destino.rename(anterior)
    tmp.rename(destino)
    if anterior.exists(): shutil.rmtree(anterior)

spark.read.parquet(str(BRONZE / "vendas")).createOrReplaceTempView("bronze_vendas")
spark.read.parquet(clientes_path).createOrReplaceTempView("bronze_clientes")
spark.read.parquet(produtos_path).createOrReplaceTempView("bronze_produtos")

spark.sql("""
CREATE OR REPLACE TEMP VIEW silver_candidatas AS
WITH numeradas AS (
 SELECT *, ROW_NUMBER() OVER (PARTITION BY venda_id ORDER BY ingestao_em DESC, arquivo_origem DESC) AS rn
 FROM bronze_vendas
), tipadas AS (
 SELECT venda_id, cliente_id, produto_id, quantidade,
        CAST(preco_unitario AS DECIMAL(14,2)) AS preco_unitario,
        TRY_CAST(data_venda AS TIMESTAMP) AS data_venda,
        ingestao_em, arquivo_origem
 FROM numeradas WHERE rn = 1
)
SELECT v.*, c.nome_cliente, c.uf, p.nome_produto, p.categoria,
       CAST(v.quantidade * v.preco_unitario AS DECIMAL(18,2)) AS valor_total,
       CASE WHEN v.venda_id IS NULL THEN 'ID_VENDA_NULO'
            WHEN v.cliente_id IS NULL OR c.cliente_id IS NULL THEN 'CLIENTE_INVALIDO'
            WHEN v.produto_id IS NULL OR p.produto_id IS NULL THEN 'PRODUTO_INVALIDO'
            WHEN v.quantidade IS NULL OR v.quantidade <= 0 THEN 'QUANTIDADE_INVALIDA'
            WHEN v.preco_unitario IS NULL OR v.preco_unitario <= 0 THEN 'PRECO_INVALIDO'
            WHEN v.data_venda IS NULL THEN 'DATA_INVALIDA'
            ELSE NULL END AS motivo_rejeicao
FROM tipadas v
LEFT JOIN bronze_clientes c ON v.cliente_id = c.cliente_id
LEFT JOIN bronze_produtos p ON v.produto_id = p.produto_id
""")

publicar_parquet(spark.sql("SELECT * FROM silver_candidatas WHERE motivo_rejeicao IS NULL"), SILVER / "vendas")
publicar_parquet(spark.sql("SELECT * FROM silver_candidatas WHERE motivo_rejeicao IS NOT NULL"), SILVER / "quarentena_vendas")
print("Silver válidas:", spark.read.parquet(str(SILVER / "vendas")).count())
print("Quarentena:", spark.read.parquet(str(SILVER / "quarentena_vendas")).count())


Silver válidas: 1000000
Quarentena: 0


## 6. Gold — indicadores de negócio em Spark SQL
Produzimos tabelas analíticas de faturamento diário, desempenho por categoria e por UF, com dados originados exclusivamente da Silver.


In [6]:
spark.read.parquet(str(SILVER / "vendas")).createOrReplaceTempView("silver_vendas")
consultas_gold = {
 "faturamento_diario": """SELECT CAST(data_venda AS DATE) AS dia,
    COUNT(*) AS total_vendas, SUM(quantidade) AS itens_vendidos,
    CAST(SUM(valor_total) AS DECIMAL(20,2)) AS faturamento,
    CAST(AVG(valor_total) AS DECIMAL(20,2)) AS ticket_medio
    FROM silver_vendas GROUP BY CAST(data_venda AS DATE)""",
 "vendas_categoria": """SELECT categoria, COUNT(*) AS total_vendas,
    SUM(quantidade) AS itens_vendidos,
    CAST(SUM(valor_total) AS DECIMAL(20,2)) AS faturamento
    FROM silver_vendas GROUP BY categoria""",
 "vendas_uf": """SELECT uf, COUNT(*) AS total_vendas,
    CAST(SUM(valor_total) AS DECIMAL(20,2)) AS faturamento
    FROM silver_vendas GROUP BY uf"""
}
for nome, sql in consultas_gold.items():
    publicar_parquet(spark.sql(sql), GOLD / nome)
    print("Gold publicada:", nome)

spark.read.parquet(str(GOLD / "vendas_categoria")).orderBy(F.desc("faturamento")).show(truncate=False)


Gold publicada: faturamento_diario
Gold publicada: vendas_categoria
Gold publicada: vendas_uf
+-----------+------------+--------------+------------+
|categoria  |total_vendas|itens_vendidos|faturamento |
+-----------+------------+--------------+------------+
|Esporte    |250000      |750000        |145973147.50|
|Eletronicos|250000      |750000        |145972870.00|
|Casa       |250000      |750000        |145972222.50|
|Moda       |250000      |750000        |145971206.85|
+-----------+------------+--------------+------------+



## 7. Testes de qualidade e reconciliação
Estas verificações demonstram integridade dos volumes, unicidade da chave de venda e consistência financeira entre Silver e Gold. O total de vendas só será 1 milhão se todos os lotes tiverem sido publicados e consumidos.


In [7]:
bronze = spark.read.parquet(str(BRONZE / "vendas"))
silver = spark.read.parquet(str(SILVER / "vendas"))
quarentena = spark.read.parquet(str(SILVER / "quarentena_vendas"))
gold_cat = spark.read.parquet(str(GOLD / "vendas_categoria"))

n_bronze, n_silver, n_quarentena = bronze.count(), silver.count(), quarentena.count()
print({"bronze": n_bronze, "silver": n_silver, "quarentena": n_quarentena})
assert n_bronze == N_VENDAS, "Nem todas as vendas chegaram à Bronze"
assert n_silver + n_quarentena == N_VENDAS, "Divergência entre Silver e quarentena"
assert silver.select("venda_id").distinct().count() == n_silver, "Venda duplicada na Silver"
assert silver.filter("valor_total <= 0 OR data_venda IS NULL").count() == 0
soma_silver = silver.agg(F.sum("valor_total")).first()[0]
soma_gold = gold_cat.agg(F.sum("faturamento")).first()[0]
assert soma_silver == soma_gold, "Faturamento da Gold diverge da Silver"
print("Validações concluídas. Faturamento:", soma_gold)


{'bronze': 1000000, 'silver': 1000000, 'quarentena': 0}
Validações concluídas. Faturamento: 583889446.85


## 8. Operação, limitações e próximos passos

- **Checkpoint:** preserva o progresso do streaming entre execuções; não apague checkpoints isoladamente enquanto mantém os dados.
- **Reexecução:** o produtor pula arquivos existentes e o streaming reconhece os arquivos já consumidos. A Silver e Gold podem ser reconstruídas a partir da Bronze.
- **Evolução de esquema:** o esquema JSON é explícito; novos campos exigem revisão dos contratos e das consultas.
- **Sem Delta Lake:** não há `MERGE`, controle de concorrência ou garantias ACID na publicação das tabelas. A estratégia de troca de diretórios é somente para uso local e didático.
- **Produção:** considere armazenamento de objetos, catálogo de tabelas, métricas de latência, tratamento de arquivos malformados, compactação e formato transacional como Iceberg/Hudi/Delta conforme os requisitos.
- **Fluxo realmente simultâneo:** execute `gerar_lotes(intervalo=10)` em outro processo Spark/produtor, enquanto o consumidor permanece ativo com `trigger(processingTime="10 seconds")`; o notebook usa `availableNow` para execução reproduzível.

**Reinício completo (destrutivo):** pare consultas ativas e execute `shutil.rmtree(BASE)`; depois reinicie o notebook do início. Não execute isso para retomar processamento.
